In [1]:
import json
import yaml
import requests
import time
from pprint import pprint
from mstrio.connection import Connection
from mstr_robotics.mstr_classes import MstrGlobal, MdSearches
from mstr_robotics.redis_db import  RedisBiAnalysis,RedisMstrJson
from mstr_robotics._connectors import MstrApi
from mstr_robotics._helper import Misc
from mstr_robotics.read_out_prj_obj import ReadGen
from mstr_robotics.prepare_ai_data import ExportMstrMd
from mstrio.api import user_hierarchies
import os
import re
from openai import OpenAI
from dotenv import load_dotenv

i_md_searches=MdSearches()
i_mstr_global=MstrGlobal()
i_mstr_api=MstrApi()
i_redis_mstr_json=RedisMstrJson()
i_export_mstr_md=ExportMstrMd()
i_read_gen=ReadGen()
i_msic=Misc()

env_file="..\\config\\streamlit.env"
load_dotenv(env_file)

with open('..\\config\\mstr_redis_y.yml', 'r') as openfile:
    mstr_redis_y = yaml.safe_load(openfile)

def grp_objtype_for_sml(obj_def_d_l):
    sml_mstr_obj_def_d_l={"dataset":{"obj_l":[],"ref_l":['https://github.com/semanticdatalayer/SML/blob/main/sml-reference/dataset.md']}, 
                          "dimension":{"obj_l":[],"ref_l":[]}}
    sml_mstr_type_d_l={"dataset":["logical_table"],
                        "dimension":["metric"]}

    for sml_md in sml_mstr_type_d_l:
        for ob in obj_def_d_l:
            
            if "subType" in ob.keys():
                if ob["subType"] in sml_mstr_type_d_l[sml_md]:
                    sml_mstr_obj_def_d_l[sml_md]["obj_l"].append(ob)
            
    return sml_mstr_obj_def_d_l

def bld_model_relations (fact_table_def): 
    relationships_d_l = []
    dimensions_l=[]
    for att in fact_table_def["attributes"]:
        rel_def_d={}
        rel_def_d["unique_name"]=fact_table_def["id"]
        rel_def_d["from"]={"dataset":fact_table_def["name"],"join_colums":["day_date"]}
        rel_def_d["to"]={"dimension":att["id"],"level":att["name"]}
        relationships_d_l.append(rel_def_d.copy())
        dimensions_l.append(att["id"])
    return {"relationships_d_l": relationships_d_l, "dimensions_l": dimensions_l}



################################
############recrusive Hier##########
def _collect_parents(attr_id, child_to_parents, visited=None):
    if visited is None:
        visited = set()
    if attr_id in visited:
        return []
    visited.add(attr_id)

    parents = child_to_parents.get(attr_id, [])
    all_ancestors = []
    for p in parents:
        all_ancestors.append(p)
        all_ancestors.extend(_collect_parents(p["objectId"], child_to_parents, visited))
    return all_ancestors

def get_all_parents_for_dims(dim_l, attribute_d_l):
    """
    For each attribute in dim_l, recursively traverses the relationship
    hierarchy in attribute_d_l and collects all parent (ancestor) attributes.

    Returns:
        dict keyed by dim id, each value is a list starting with the dim itself
        followed by ancestor dicts, each with keys: objectId, subType, name
    """
    # Build child_id -> list of parent info from all attribute relationships
    child_to_parents = {}
    for attr in attribute_d_l:
        for rel in attr.get("relationships", []):
            child_id = rel["child"]["objectId"]
            parent_info = rel["parent"]
            if child_id not in child_to_parents:
                child_to_parents[child_id] = []

            # Avoid duplicate parents
            if not any(p["objectId"] == parent_info["objectId"] for p in child_to_parents[child_id]):
                child_to_parents[child_id].append(parent_info)

    result = {}
    for dim in dim_l:
        ancestors = _collect_parents(dim["id"], child_to_parents)
        # Deduplicate while preserving order
        seen = set()
        unique_ancestors = []
        for a in ancestors:
            if a["objectId"] not in seen:
                seen.add(a["objectId"])
                unique_ancestors.append(a)
        dim_self = {"objectId": dim["id"], "subType": "attribute", "name": dim["name"]}
        result[dim["id"]] = [dim_self] + unique_ancestors

    return result

In [2]:
redis_con_d=mstr_redis_y["redis_env_d"]["redis_dev"]
project_prefix=mstr_redis_y["project_prefix"]
prefix_map=mstr_redis_y["prefix_map"]
searches_used_in_prp_d_l=mstr_redis_y["searches_used_in_prp_d_l"]


## Connect to MSTR & Redis

In [3]:
with open('..\\config\\user_d.json', 'r') as openfile:
    user_d = json.load(openfile)
conn_params =  user_d["conn_params"]
conn = Connection(**conn_params)
conn.headers['Content-type'] = "application/json"

i_redis_bi_analysis = RedisBiAnalysis( 
    host=redis_con_d["host"],
    port=redis_con_d["port"],
    password=redis_con_d["password"],
    username=redis_con_d["username"],
    decode_responses=redis_con_d["decode_responses"]
)
conn.select_project("B7CA92F04B9FAE8D941C3E9B7E0CD754")

Connection to Strategy One Intelligence Server has been established.
Project selected in Connection object:
Project object named: 'MicroStrategy Tutorial' with ID: 'B7CA92F04B9FAE8D941C3E9B7E0CD754'


## Fetch MSTR objects

### report tables

In [ ]:
redis_env_p="mstr_dev"
redis_pre_obj="TABLE"
table_key_l=[]
table_d_l=[{"id":"24C30AD611D5AEC9C000E38A4CC5F24F","name":"lu_day"},
          {"id":"8D67933211D3E4981000E787EC6DE8A4","name":"lu_call_ctr"},
          {"id":"8D67933E11D3E4981000E787EC6DE8A4","name":"lu_category"},
          {"id":"8D67936811D3E4981000E787EC6DE8A4","name":"lu_employee"},
          {"id":"8D67937411D3E4981000E787EC6DE8A4","name":"lu_item"},
          {"id":"8D67938011D3E4981000E787EC6DE8A4","name":"lu_month"},
          {"id":"8D6793A411D3E4981000E787EC6DE8A4","name":"lu_quarter"},
          {"id":"8D6793C211D3E4981000E787EC6DE8A4","name":"lu_year"},
          {"id":"8D6793AA11D3E4981000E787EC6DE8A4","name":"lu_region"},
          {"id":"8D6793B611D3E4981000E787EC6DE8A4","name":"lu_subcateg"},
          {"id":"8D6793CE11D3E4981000E787EC6DE8A4","name":"order_detail"}]
for table in table_d_l:
    oby_key=f'{redis_env_p}:{redis_pre_obj}:{table["id"]}'
    table_key_l.append(oby_key)

### Fetch Dependencies for Specific Report

In [ ]:
from mstr_robotics.redis_db import FetchItAll

# Initialize the fetch_it_all class with your Redis connection
i_fetch_it_all = FetchItAll(i_redis_bi_analysis)

# Define the root object key for the report definition you want to fetch
root_object_key = "mstr_dev:REPORT_DEFINITION:66FFEB6E49634F501160D2A4FB9BD78A"

# Fetch all dependencies recursively
all_dependencies = i_fetch_it_all.fetch_all_objects_recursively(
    root_object_l=[root_object_key],
    recursive_fg=True,
    batch_size=100
)

In [ ]:
#prepare MD
direct_obj_check_d_l=[]
obj_type_set=set()
obj_to_export_d_l=[]
for obj in all_dependencies:
    try:
        if "information" in obj["definition"].keys():
            obj_type_set.add(obj["definition"]["information"]["subType"])
            if obj["definition"]["information"]["subType"] in ["report_grid","agg_metric"]:
                direct_obj_check_l=i_fetch_it_all.fetch_all_objects_recursively(root_object_l=[obj["obj_key"]]
                                             , recursive_fg=False)
                direct_obj_check_d_l.extend(direct_obj_check_l)
            if obj["definition"]["information"]["subType"] in ["report_grid","agg_metric"]:
                obj_to_export_d_l.extend([obj["obj_key"]])
        else:
            obj_type_set.add(obj["definition"]["subType"])
            if obj["definition"]["subType"] in ["metric","agg_metric"]:
                direct_obj_check_l=i_fetch_it_all.fetch_all_objects_recursively(root_object_l=[obj["obj_key"]]
                                             , recursive_fg=False)
                direct_obj_check_d_l.extend(direct_obj_check_l)
            if obj["definition"]["subType"] in ["metric","filter","prompt","role_transformation"]:
                obj_to_export_d_l.extend([obj["obj_key"]])       
    except:
        print("rreee")
        print(obj["obj_key"])

schema_obj_l=[]
for obj in direct_obj_check_d_l:
    try:
        if "information" in obj["definition"].keys():
            if obj["definition"]["information"]["subType"] in ["attribute","agg_metric"]:
                schema_obj_l.append(obj["obj_key"])
        
        else:
            #print(obj["definition"]["subType"])
            if obj["definition"]["subType"] in ["attribute","fact","hierarchy"]:
                schema_obj_l.append(obj["obj_key"])
    except:
        print("rreee")

schema_obj_l=i_msic.rem_dbl_in_l(schema_obj_l)
#schema_obj_d_l = remove_duplicates_by_obj_id(schema_obj_d_l)
obj_to_export_d_l.extend(schema_obj_l)
obj_to_export_d_l.extend(table_key_l)
obj_to_export_d_l
obj_id_l=[]

for obj in obj_to_export_d_l:
    obj_id_l.append(obj.split(":")[2])
obj_def_d_l=i_read_gen.get_proj_obj_def_by_id_l(conn,obj_id_l)
sml_mstr_obj_def_d_l=grp_objtype_for_sml(obj_def_d_l)



In [ ]:
from mstr_robotics._paths import REPO_ROOT, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
fact_table_l=["8D6793CE11D3E4981000E787EC6DE8A4"]
relationships_d_l=[]
dimensions_l=[]
for obj in obj_def_d_l:
    try:
        if obj["id"] in fact_table_l:
            model_rel_d=bld_model_relations(fact_table_def=obj)
            relationships_d_l.extend(model_rel_d["relationships_d_l"])
            dimensions_l.extend(model_rel_d["dimensions_l"])
    except:
        pass
    
model_d = {
    "unique_name": "TutorialModel",
    "object_type": "model",
    "label": "MicroStrategy Tutorial Model",
    "visible": True,
    "relationships": relationships_d_l,
    "dimensions": dimensions_l
}

with open(str(PYTHON_IO / "output_files" / "SML" / "model.md"), 'w', encoding='utf-8') as f:
    yaml.dump(model_d, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

In [ ]:
dataset_cols_l=["physicalTable", "isPartOfPartition","primaryDataSource", "secondaryDataSources","primaryLocale","name"]
mapped_schema_l=["attributes","facts","name"]
tableKey_l=["tableKey","name"]

sml_mstr_obj_def_d_l={"dataset":{"dataset_rag_l":[],"ref_l":['https://github.com/semanticdatalayer/SML/blob/main/sml-reference/dataset.md']} 
                      ,"mapped_schema_l":{"obj_l":[],"ref_l":[]}
                      ,"tablekeys_l":{"obj_l":[],"ref_l":[]}
                      ,"dimension":{"obj_l":[],"ref_l":[]}
                      }

sml_dataset_rag_l=[]
sml_dataset_expressions_rag_l=[]
sml_tablekeys_rag_l=[]
for obj in obj_def_d_l:
    if "physicalTable" in obj.keys():
        sml_mstr_obj_def_d_l["dataset"]["dataset_rag_l"].append(Misc().select_dict_cols(obj, dataset_cols_l))
        sml_mstr_obj_def_d_l["mapped_schema_l"]["obj_l"].append(Misc().select_dict_cols(obj, mapped_schema_l))
        sml_mstr_obj_def_d_l["tablekeys_l"]["obj_l"].append(Misc().select_dict_cols(obj, tableKey_l))



In [ ]:
# dimension / attribute extraction
attribute_d_l=[]
all_child_l=[]
all_parent_l=[]
dim_l=[]
child_all_parent={}

for obj in obj_def_d_l:
    try:
        if obj["subType"]=="attribute":
            attribute_d_l.append(obj)
    except:
        pass

for a in attribute_d_l:
    for r in a["relationships"]:
        all_parent_l.append(r["parent"]["objectId"])
        all_child_l.append(r["child"]["objectId"])
dimId_l=set(all_child_l)-set(all_parent_l)
dimId_l=list(dimId_l)

for a in attribute_d_l:
    if a["id"] in dimId_l:
        dim_l.append({"id":a["id"], "name":a["name"]})

dim_parents = get_all_parents_for_dims(dim_l, attribute_d_l)
for dim_name, parents in dim_parents.items():
    child_all_parent[dim_name] = {"level_attributes": parents, "secondary_attriutes": []}
child_all_parent

In [ ]:
# Extract level attributes from logical tables
level_att_l = []
for obj in obj_def_d_l:
    if obj.get("subType") != "logical_table":
        continue
    
    table_id = obj["id"]
    table_name = obj["name"]
    
    for att in obj.get("attributes", []):
        for form in att.get("forms", []):
            lookup_id = form.get("lookupTable", {}).get("objectId")
            if lookup_id == table_id:
                level_att_d = {
                    "table_id": table_id,
                    "table_name": table_name,
                    "att_id": att["id"],
                    "att_name": att["name"]
                }
                level_att_l.append(level_att_d)

# Remove duplicates
level_att_l = i_msic.rem_dbl_dict_in_l(level_att_l)

# Collect all child attribute IDs from relationships
child_att_l = []
for la in level_att_l:
    for a in attribute_d_l:
        if la["att_id"] == a["id"]:
            for r in a["relationships"]:
                child_att_l.append(r["child"]["objectId"])

child_att_l = list(set(child_att_l))



In [ ]:
sml_dimensions_d=child_all_parent.copy()
#print(sml_dimensions_d)
entry_att_hier_d={"96ED3EC811D5B117C000E78A4CC5F24F":"A00D4B5546C01F58A8691CB440BD8C41",
                  "8D679D4211D3E4981000E787EC6DE8A4":"FEA63419412BE1CA5ABA5B9632D6AB38",
                  "8D679D3F11D3E4981000E787EC6DE8A4":"9981A8614C30B583A641CCA84042F0A0"}
for dim in sml_dimensions_d.keys():
    user_hierarchy_def=user_hierarchies.get_user_hierarchy(connection=conn,project_id=conn.project_id,
                                        id=entry_att_hier_d[dim]).json()
    level_att_fin_l=[]
    for att in user_hierarchy_def["attributes"]:
        level_att_fin_l.append(att["objectId"])

    for att in sml_dimensions_d[dim]["level_attributes"]:
        #print(att["objectId"])
        if att["objectId"] not in level_att_fin_l:
            
            sml_dimensions_d[dim]["level_attributes"].remove(att)
            sml_dimensions_d[dim]["secondary_attriutes"].append(att)

with open('..\\config\\sml\\template_dim.yml', 'r') as openfile:
    dims_y = yaml.safe_load(openfile)

#pprint(dims_y)

for d in dims_y:
    #print(d)
    d_l=[]
    for dim in sml_dimensions_d:
        if d ==dim:
            #print(dim)
            for level_att in sml_dimensions_d[dim]["level_attributes"]:
                d_l.append(d)
                dims_y[d]["hierarchies"]["levels"].append({d:level_att["name"]})

dims_y